In [1]:
import pandas as pd

In [2]:
import numpy as np

In [ ]:
from sklearn.preprocessing import RobustScaler

In [11]:
CSV_PATH = "./dataset/raw_loan_dataset.csv"

In [8]:
OUT_PATH = "./dataset/clean_loan_dataset.csv"

#### Load + initial snapshot

In [15]:
df = pd.read_csv("C:/Users/Lenovo/Documents/test three python/raw_loan_dataset.csv")

#### INITIAL HEAD

In [16]:
print(df.head())

    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  


#### INITIAL INFO

In [17]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Approved          103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB
None


#### MISSING VALUES

In [18]:
print(df.isnull().sum())

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64


#### Clean currency formatting

In [21]:
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)

df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)

In [22]:
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,Yes,No,No
1,96482.0,524.0,15.0,11200.0,Y,No,Yes
2,28478.0,NaN,5.4,12100.0,No,No,Yes
3,25851.0,561.0,17.6,7000.0,Yes,No,Yes
4,38396.0,527.0,9.8,10700.0,No,No,Approved


#### Normalize categorical typos BEFORE imputation

In [48]:
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}

In [49]:
df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

In [50]:
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,Yes,No,No
1,96482.0,524.0,15.0,11200.0,Yes,No,Yes
2,28478.0,638.0,5.4,12100.0,No,No,Yes
3,25851.0,561.0,17.6,7000.0,Yes,No,Yes
4,38396.0,527.0,9.8,10700.0,No,No,Yes


#### Impute missing values

In [51]:
df["Income"] = df["Income"].fillna(df["Income"].median())

df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())

df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())

df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())

df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])

df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])

In [52]:
before = df.shape[0]

#### Remove duplicates

In [53]:
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")


Dropped duplicates: 100 -> 100 rows


#### IQR capping on numeric columns

In [54]:
### clip extreme values to the IQR fence

In [55]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

In [56]:
low_income,  high_income  = iqr_fun(df["Income"])

low_credit,  high_credit  = iqr_fun(df["CreditScore"])

low_loan,    high_loan    = iqr_fun(df["LoanAmount"])

low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

In [57]:
df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)

df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)

df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)

df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)

In [58]:
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810.0,537.0,1.1,25800.0,Yes,No,No
1,96482.0,524.0,15.0,11200.0,Yes,No,Yes
2,28478.0,638.0,5.4,12100.0,No,No,Yes
3,25851.0,561.0,17.6,7000.0,Yes,No,Yes
4,38396.0,527.0,9.8,10700.0,No,No,Yes


#### Label encoding (Yes/No → 0/1)

In [59]:
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)

#### CLASS DISTRIBUTION (after label encoding)

In [60]:
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))

1    66
0    34
Name: Approved, dtype: int64
1    0.66
0    0.34
Name: Approved, dtype: float64


#### Class balance check

In [61]:
### L3/L5: imbalanced classes can make Accuracy misleading

In [62]:
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")



Class balance OK for baseline Accuracy (both classes well represented).


#### Feature engineering (no leakage from label)

In [70]:
df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)

#### StandardScaler on numeric features

In [71]:
###  scale after outlier capping so mean/std are not skewed by extremes
# Exclude label (Approved) and already-binary columns from scaling

In [72]:
binary_cols = {"HasCollateral", "PreviousDefaults", "Approved"}
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in binary_cols]

In [ ]:
scaler = RobustScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

#### Final snapshot + save


In [74]:
###  Features (X) = all columns except Approved; label (y) = Approved

#### FINAL HEAD 

In [75]:
print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  1.260277    -1.191820        -1.656317    0.019897              1   
1  0.835658    -1.327610         0.336223   -0.971972              1   
2 -1.506631    -0.136835        -1.039920   -0.910830              0   
3 -1.597114    -0.941130         0.708929   -1.257305              1   
4 -1.165022    -1.296274        -0.409187   -1.005941              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0     -0.782986               4.229042  
1                 0         1     -1.549997              -0.268880  
2                 0         1      0.407068              -0.424145  
3                 0         1     -0.569589              -0.724750  
4                 0         1     -0.519571              -0.512022  


#### FINAL INFO

In [76]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 100 non-null    float64
 1   CreditScore            100 non-null    float64
 2   EmploymentYears        100 non-null    float64
 3   LoanAmount             100 non-null    float64
 4   HasCollateral          100 non-null    int32  
 5   PreviousDefaults       100 non-null    int32  
 6   Approved               100 non-null    int32  
 7   DebtToIncome           100 non-null    float64
 8   IncomePerYearEmployed  100 non-null    float64
dtypes: float64(6), int32(3)
memory usage: 6.6 KB
None


#### FINAL MISSING VALUES

In [77]:
print(df.isnull().sum())

Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
IncomePerYearEmployed    0
dtype: int64


In [82]:
OUT_PATH = "C:/Users/Lenovo/Documents/test three python/dataset/clean_loan_dataset.csv"

df.to_csv(OUT_PATH, index=False)

In [83]:
print(f"\nSaved cleaned dataset to {OUT_PATH}")


Saved cleaned dataset to C:/Users/Lenovo/Documents/test three python/dataset/clean_loan_dataset.csv
